# Linux User Management (Educational Notebook)
This notebook covers how Linux represents users and groups, where that information is stored, and the everyday commands for creating, modifying, and removing accounts. It sets up the vocabulary (UID, GID, owner) that the permissions lessons later in `week3/` build on.

## 1. What Is a User?

Every process and every file on a Linux system is associated with a **user**, identified internally not by name but by a numeric **UID** (User ID). The username is just a human-friendly label the system maps to that number.

```bash
id                # show your own UID, GID, and group memberships
id alice            # show the same information for another user
whoami                # print just the current username
```


## 2. Types of Users

| Type | UID range (typical) | Purpose |
|---|---|---|
| **root** | `0` | The superuser - bypasses normal permission checks almost entirely |
| **System users** | `1-999` (varies by distro) | Non-interactive accounts owned by services and daemons (e.g. `www-data`, `sshd`, `mysql`), not meant for human login |
| **Regular users** | `1000+` (varies by distro) | Ordinary human accounts created for people to log in with |

Giving each service its own dedicated system user - rather than running everything as root - limits the damage a compromised service can do, since that user typically owns only the files it needs.


## 3. Groups

A **group** is a named collection of users that can be granted permissions collectively, rather than one user at a time.

- Every user has exactly one **primary group** (used, among other things, as the default group for files they create).
- A user can additionally belong to any number of **supplementary (secondary) groups**, granting access to whatever those groups are permitted.

```bash
groups              # list the groups the current user belongs to
groups alice           # list the groups a specific user belongs to
id -Gn alice              # the same information via id, group names only
```


## 4. Where User and Group Information Lives

User and group accounts are ordinary text files under `/etc/`, each with a fixed, colon-separated field format.

**`/etc/passwd`** - one line per user:

```
alice:x:1000:1000:Alice Smith:/home/alice:/bin/bash
```

| Field | Meaning |
|---|---|
| 1 | Username |
| 2 | Password placeholder (`x` means the real hash lives in `/etc/shadow`) |
| 3 | UID |
| 4 | Primary GID |
| 5 | Comment/full name (the "GECOS" field) |
| 6 | Home directory |
| 7 | Login shell |

**`/etc/shadow`** - the actual (hashed) passwords and password-aging policy, readable only by root, since `/etc/passwd` itself must remain world-readable for many tools to resolve UIDs to names.

**`/etc/group`** - one line per group, similarly colon-separated: group name, password placeholder, GID, and a comma-separated list of supplementary members.

```bash
cat /etc/passwd | grep alice
cat /etc/group | grep alice
sudo cat /etc/shadow | grep alice     # requires root -- the file is deliberately restricted
getent passwd alice                     # the preferred, NSS-aware way to look up a user (also works with LDAP/etc, not just local files)
```


## 5. Creating Users

```bash
sudo useradd alice                      # create a user with system defaults (may not create a home directory, depending on distro)
sudo useradd -m alice                     # create the user AND their home directory
sudo useradd -m -s /bin/bash alice          # also set their login shell explicitly
sudo useradd -m -G sudo,developers alice      # also add them to supplementary groups at creation time

sudo adduser alice                              # Debian/Ubuntu's higher-level, interactive wrapper around useradd
                                                    # -- prompts for a password and details, and is generally friendlier for manual use
```

`useradd` is the lower-level, script-friendly command found on every distribution; `adduser` (where available) is a more interactive, distro-specific convenience wrapper around it.


## 6. Modifying Users

```bash
sudo usermod -aG developers alice      # ADD alice to a supplementary group (always use -a here -- see warning below)
sudo usermod -s /bin/zsh alice           # change a user's login shell
sudo usermod -d /new/home -m alice         # change (and move) a user's home directory
sudo usermod -l newname alice                # rename a user's login name
sudo usermod -L alice                          # lock the account (disable password login)
sudo usermod -U alice                            # unlock it again
```

**Warning:** `usermod -G developers alice` (without `-a`) *replaces* alice's entire supplementary group list with just `developers`, silently removing every other group membership. Always combine `-a` (append) with `-G` unless you genuinely intend to replace the whole list.


## 7. Deleting Users

```bash
sudo userdel alice           # remove the user account, leaving their home directory and files in place
sudo userdel -r alice           # also remove their home directory and mail spool
```

Before deleting a user, check whether they own files elsewhere on the system (`find / -user alice`) - those files aren't removed and will be left owned by a now-nonexistent UID, which can be confusing later.


## 8. Creating and Managing Groups

```bash
sudo groupadd developers          # create a new group
sudo groupmod -n devs developers    # rename a group
sudo groupdel devs                    # delete a group (fails if it's still someone's primary group)

sudo gpasswd -a alice developers        # an alternative way to add a user to a group
sudo gpasswd -d alice developers          # remove a user from a group
```


## 9. Passwords and Account Aging

```bash
passwd                    # change your own password
sudo passwd alice            # set/change another user's password (as root)
sudo passwd -l alice           # lock a password (prepends a marker to the hash, disabling password login)
sudo passwd -u alice             # unlock it again
sudo passwd -e alice               # expire a password immediately, forcing a change at next login

chage -l alice              # list a user's password aging information
sudo chage -M 90 alice         # require a password change at least every 90 days
sudo chage -E 2026-12-31         # set an account expiration date
```

Note `passwd -l`/`-u` lock/unlock the *password*, while `usermod -L`/`-U` achieves the same underlying effect - the two commands overlap in purpose.


## 10. Switching Users and Privilege Escalation

```bash
su alice              # switch to another user (keeps the current shell's environment)
su - alice               # switch to another user AND start a fresh login shell (their environment, their $HOME, etc.)
su -                        # switch to root, with root's own environment

sudo <command>          # run a single command as root (or another user), typically after entering YOUR OWN password
sudo -u alice <command>    # run a command as a specific user other than root
sudo -i                      # start an interactive root login shell

sudo visudo                    # safely edit /etc/sudoers (validates syntax before saving, preventing you from locking yourself out)
```

`su` requires knowing the **target** account's password; `sudo` instead checks whether *your own* account has been granted permission (via `/etc/sudoers` or a file under `/etc/sudoers.d/`) and, if so, asks for *your* password instead. This is why `sudo` is generally preferred for day-to-day administration - it doesn't require sharing the root password at all.


## 11. Useful Query Commands

```bash
id                    # your own UID/GID/groups
id alice                 # another user's UID/GID/groups
whoami                     # current username only
groups alice                  # groups a user belongs to
getent passwd alice             # look up a user (works with local files, LDAP, etc.)
getent group developers           # look up a group the same way
who                                  # who is currently logged in
w                                       # who is logged in, plus what they're running
last                                       # a log of recent logins
```


## Hands-on

Try these on your own system (a virtual machine is safest for practicing account changes):

```bash
sudo useradd -m -s /bin/bash testuser
sudo passwd testuser
id testuser
cat /etc/passwd | grep testuser

sudo groupadd testgroup
sudo usermod -aG testgroup testuser
groups testuser

sudo passwd -l testuser
sudo passwd -u testuser

sudo userdel -r testuser
sudo groupdel testgroup
```

At each step, check `/etc/passwd`, `/etc/group`, and the output of `id testuser` to see exactly what changed.


## Review Questions

1. What is a UID, and why does the system use it internally instead of the username?
2. What distinguishes a system user from a regular user, and why do services typically get their own dedicated user rather than running as root?
3. Name the four fields you'd need to combine to fully identify a user account across `/etc/passwd` and `/etc/shadow`.
4. Why is `/etc/shadow` restricted to root, when `/etc/passwd` is world-readable?
5. What's the difference between a user's primary group and their supplementary groups?
6. Why is `usermod -G developers alice` dangerous compared to `usermod -aG developers alice`?
7. What does `userdel -r` do differently from plain `userdel`?
8. What's the practical difference between `su alice` and `su - alice`?
9. Why is `sudo` generally considered safer for day-to-day administration than sharing the root password via `su`?
10. Why should you edit `/etc/sudoers` with `visudo` instead of a plain text editor?


# Cheat Sheet

```
Identify:
  id [user]      whoami      groups [user]      getent passwd [user]      getent group [group]

Account files:
  /etc/passwd    username:x:UID:GID:comment:home:shell
  /etc/shadow    hashed passwords + aging policy (root-only)
  /etc/group     groupname:x:GID:member1,member2

Create/modify/delete users:
  useradd -m -s /bin/bash -G group1,group2 <user>
  usermod -aG <group> <user>       (ALWAYS use -a when adding a group)
  usermod -s <shell> | -d <dir> -m | -l <newname> | -L | -U
  userdel <user>          userdel -r <user>   (also remove home dir)

Groups:
  groupadd <group>      groupmod -n <new> <old>      groupdel <group>
  gpasswd -a <user> <group>      gpasswd -d <user> <group>

Passwords/aging:
  passwd [user]      passwd -l|-u|-e [user]
  chage -l <user>      chage -M <days> <user>      chage -E <date> <user>

Switch users / escalate:
  su <user>      su - <user>      su -
  sudo <cmd>      sudo -u <user> <cmd>      sudo -i      sudo visudo
```
